<a href="https://colab.research.google.com/github/daramnandini/SmartCut_AI/blob/main/SmartCut_AI.ipynb." target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rembg onnxruntime gradio pillow numpy

import gradio as gr
import numpy as np
import io
import os
import tempfile

from PIL import (
    Image, ImageFilter, ImageEnhance,
    ImageChops
)

from rembg import remove, new_session

print("Loading SMARTCUT AI Model...")

session = new_session("u2net")

print("AI Model Ready!")

def remove_background(image):

    if image is None:
        raise gr.Error("Please upload an image.")

    image = image.convert("RGBA")

    output = remove(
        image,
        session=session,
        alpha_matting=True,
        alpha_matting_foreground_threshold=240,
        alpha_matting_background_threshold=10,
        alpha_matting_erode_size=10
    )

    return output.convert("RGBA")

def smartcut_process(
    image,
    background_mode,
    background_color,
    custom_background,
    subject_size,
    horizontal_position,
    vertical_position,
    rotation,
    brightness,
    contrast,
    shadow
):

    if image is None:
        raise gr.Error("Please upload an image first!")

    original = image.convert("RGB")

    foreground = remove_background(original)
    foreground.save(
        "/tmp/SmartCut_Transparent.png",
        "PNG"
    )

    W, H = original.size

    if background_mode == "Solid Color":

        background = Image.new(
            "RGBA",
            (W, H),
            background_color
        )

    elif background_mode == "Custom Image":

        if custom_background is None:
            raise gr.Error("Upload a custom background.")

        background = custom_background.convert("RGB")

        background = background.resize(
            (W, H),
            Image.Resampling.LANCZOS
        ).convert("RGBA")

    elif background_mode == "Nature":

        background = Image.new(
            "RGBA", (W, H), "#75c9a5"
        )

    elif background_mode == "Sky":

        background = Image.new(
            "RGBA", (W, H), "#87CEEB"
        )

    elif background_mode == "Professional":

        background = Image.new(
            "RGBA", (W, H), "#e5e7eb"
        )

    else:

        background = Image.new(
            "RGBA", (W, H), "#ffffff"
        )

    # Subject resizing
    scale = subject_size / 100

    new_w = max(1, int(foreground.width * scale))
    new_h = max(1, int(foreground.height * scale))

    subject = foreground.resize(
        (new_w, new_h),
        Image.Resampling.LANCZOS
    )
    subject = subject.rotate(
        rotation,
        resample=Image.Resampling.BICUBIC,
        expand=True
    )
    rgb = Image.new("RGB", subject.size, "white")
    rgb.paste(subject, mask=subject.getchannel("A"))

    rgb = ImageEnhance.Brightness(
        rgb
    ).enhance(brightness / 100)

    rgb = ImageEnhance.Contrast(
        rgb
    ).enhance(contrast / 100)

    subject.putdata([
        (*pixel, alpha)
        for pixel, alpha in zip(
            rgb.getdata(),
            subject.getchannel("A").getdata()
        )
    ])

    x = int((W - new_w) / 2 +
            horizontal_position * W / 100)

    y = int((H - new_h) / 2 +
            vertical_position * H / 100)
    if shadow:

        alpha = subject.getchannel("A")

        shadow_layer = Image.new(
            "RGBA",
            (W, H),
            (0, 0, 0, 0)
        )

        shadow_mask = Image.new(
            "RGBA",
            subject.size,
            (0, 0, 0, 120)
        )

        shadow_mask.putalpha(
            alpha.point(lambda p: int(p * 0.45))
        )

        shadow_mask = shadow_mask.filter(
            ImageFilter.GaussianBlur(12)
        )

        shadow_layer.alpha_composite(
            shadow_mask,
            (x + 10, y + 12)
        )

        background.alpha_composite(shadow_layer)

    foreground_layer = Image.new(
        "RGBA", (W, H), (0, 0, 0, 0)
    )

    foreground_layer.alpha_composite(
        subject,
        (x, y)
    )

    final_image = Image.alpha_composite(
        background,
        foreground_layer
    )
    final_path = "/tmp/SmartCut_Final.png"

    final_image.convert("RGB").save(
        final_path,
        quality=95
    )

    return (
        original,
        foreground,
        final_image.convert("RGB"),
        final_path
    )


def download_transparent(image):

    if image is None:
        raise gr.Error("Please process an image first.")

    path = "/tmp/SmartCut_Transparent.png"

    image.save(path, "PNG")

    return path

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SMARTCUT AI"
) as demo:

    gr.Markdown(
        """
        # ✂️ SMARTCUT AI
        ### Intelligent Image Background Removal & Replacement System

        Upload an image, remove its background using AI,
        customize the background and download your final image.
        """
    )

    with gr.Row():

        input_image = gr.Image(
            label="Upload Original Image",
            type="pil"
        )

        with gr.Column():

            background_mode = gr.Dropdown(
                choices=[
                    "Solid Color",
                    "Custom Image",
                    "Nature",
                    "Sky",
                    "Professional"
                ],
                value="Solid Color",
                label="Background Type"
            )

            background_color = gr.ColorPicker(
                value="#87CEEB",
                label="Choose Background Colour"
            )

            custom_background = gr.Image(
                label="Upload Custom Background",
                type="pil"
            )

    gr.Markdown("## 🎨 Subject Editing Controls")

    with gr.Row():

        subject_size = gr.Slider(
            20, 150,
            value=100,
            label="Subject Size (%)"
        )

        rotation = gr.Slider(
            -180, 180,
            value=0,
            label="Rotation"
        )

    with gr.Row():

        horizontal_position = gr.Slider(
            -50, 50,
            value=0,
            label="Horizontal Position"
        )

        vertical_position = gr.Slider(
            -50, 50,
            value=0,
            label="Vertical Position"
        )

    with gr.Row():

        brightness = gr.Slider(
            50, 150,
            value=100,
            label="Brightness"
        )

        contrast = gr.Slider(
            50, 150,
            value=100,
            label="Contrast"
        )

    shadow = gr.Checkbox(
        label="Enable Realistic Shadow",
        value=False
    )

    process_button = gr.Button(
        "✂️ REMOVE BACKGROUND & CREATE IMAGE",
        variant="primary"
    )

    gr.Markdown("## 🖼️ Image Comparison")

    with gr.Row():

        original_output = gr.Image(
            label="Original Image"
        )

        transparent_output = gr.Image(
            label="AI Background Removed",
            type="pil"
        )

        final_output = gr.Image(
            label="Final SMARTCUT Image"
        )

    with gr.Row():

        transparent_download = gr.Button(
            "Download Transparent PNG"
        )

        final_download = gr.File(
            label="Download Final Image"
        )

    process_button.click(
        fn=smartcut_process,
        inputs=[
            input_image,
            background_mode,
            background_color,
            custom_background,
            subject_size,
            horizontal_position,
            vertical_position,
            rotation,
            brightness,
            contrast,
            shadow
        ],
        outputs=[
            original_output,
            transparent_output,
            final_output,
            final_download
        ]
    )

    transparent_download.click(
        fn=download_transparent,
        inputs=[transparent_output],
        outputs=gr.File(label="Transparent PNG")
    )

    gr.Markdown(
        """
        ---
        **SMARTCUT AI | Artificial Intelligence & Computer Vision**

        Powered by Deep Learning, Image Processing and Image Segmentation.
        """
    )


# ==========================================================
# LAUNCH APPLICATION
# ==========================================================

print("Starting SMARTCUT AI Application...")

demo.launch(share=True, debug=True)

Loading SMARTCUT AI Model...


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

AI Model Ready!


/tmp/ipykernel_2309/1860253963.py:266: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Starting SMARTCUT AI Application...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ff53c2b6229d901a25.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
